# Análisis exploratorio de datos

## Carga de datos

In [67]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import yfinance as yf
import math
import datetime
from scipy.stats import boxcox
from statsmodels.tsa.stattools import adfuller
from pathlib import Path


sns.set_style("darkgrid")
sns.set_context("poster")

In [68]:
pd.set_option('display.max_columns', None) # Por si se quiere ver todas las columnas del DataFrame
pd.set_option('display.max_rows', None) # Por si se quiere ver todas las filas del DataFrame
pd.set_option('display.max_colwidth', None) # Por si se quiere ver todo el contenido de las columnas del DataFrame

In [69]:
DATA_PATH_GEN_DIARIA = Path("../data/raw/ree_generacion_diaria.csv")
DATA_PATH_HORARIO = Path("../data/raw/ree_horario.csv")

if not DATA_PATH_GEN_DIARIA.exists():
    # Alternativa si se ejecuta desde la raíz del proyecto
    DATA_PATH_GEN_DIARIA = Path("data/raw/ree_generacion_diaria.csv")

if not DATA_PATH_HORARIO.exists():
    # Alternativa si se ejecuta desde la raíz del proyecto
    DATA_PATH_HORARIO = Path("data/raw/ree_horario.csv")

df_horario = pd.read_csv(DATA_PATH_HORARIO)
df_generacion = pd.read_csv(DATA_PATH_GEN_DIARIA)

print(
    f"✓ Dataset cargado correctamente con {df_horario.shape[0]} filas y {df_horario.shape[1]} columnas a partir de ree_horario.csv."
)
print(
    f"✓ Dataset cargado correctamente con {df_generacion.shape[0]} filas y {df_generacion.shape[1]} columnas a partir de ree_generacion_diaria.csv."
)
assert df_horario.shape == (
    26280,
    6,
), f"Dimensiones inesperadas: {df_horario.shape} (se esperaban 26280 filas y 6 columnas)"

assert df_generacion.shape == (
    1095,
    18,
), f"Dimensiones inesperadas: {df_generacion.shape} (se esperaban 1095 filas y 18 columnas)"

df_horario.head()

✓ Dataset cargado correctamente con 26280 filas y 6 columnas a partir de ree_horario.csv.
✓ Dataset cargado correctamente con 1095 filas y 18 columnas a partir de ree_generacion_diaria.csv.


,fecha_hora,demanda_prevista,demanda_programada,demanda_real,precio_spot,pvpc
0,2021-01-01 00:00:00+01:00,26104.667,26013.0,26151.667,50.87,NaN
1,2021-01-01 01:00:00+01:00,24822.583,24630.0,24886.250,48.19,NaN
2,2021-01-01 02:00:00+01:00,22652.667,22696.0,22685.500,44.68,NaN
3,2021-01-01 03:00:00+01:00,20786.833,20756.0,20828.250,38.50,NaN
4,2021-01-01 04:00:00+01:00,19877.917,19800.0,19877.583,36.80,NaN


In [70]:
df_generacion.head()

,fecha,gen_hidráulica,gen_nuclear,gen_carbón,gen_motores_diésel,gen_turbina_de_gas,gen_turbina_de_vapor,gen_ciclo_combinado,gen_hidroeólica,gen_eólica,gen_solar_fotovoltaica,gen_solar_térmica,gen_otras_renovables,gen_cogeneración,gen_residuos_no_renovables,gen_residuos_renovables,gen_generación_total,gen_fuel_gas
0,2021-01-01,111077.579,170966.633,5663.327,6180.245,840.614,3681.076,61416.329,15.538,240290.824,29876.523,3126.095,13234.386,54581.986,4477.651,1232.274,706661.080,NaN
1,2021-01-02,112211.332,170974.655,5611.373,7074.243,722.388,4587.264,80555.288,1.957,270599.158,34247.163,6340.881,12636.823,67513.021,4389.159,1262.089,778726.794,NaN
2,2021-01-03,116969.983,171000.851,5597.198,7036.465,993.803,4502.216,83628.611,15.593,221167.693,37134.485,7441.714,12215.001,73185.396,4896.856,1250.171,747036.037,NaN
3,2021-01-04,165965.559,170871.508,8431.686,7371.259,1328.929,3783.796,144897.990,34.367,130542.499,26307.981,2783.566,12124.513,77455.663,5155.834,1215.017,758270.167,NaN
4,2021-01-05,176184.581,170882.098,16295.678,7285.896,1006.802,4453.566,138876.901,-3.059,70360.924,29746.352,2099.695,12386.598,80838.906,5499.507,1230.110,717144.555,NaN


Se comienza analizando `df_horario`, ya que contiene la variable `demanda_real`, que es la que se busca predecir. Sin embargo, se puede observar que viene en horas, mientras que el segundo fichero viene en días, por lo que será necesario más adelante una conversión a días.

In [71]:
df_horario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26280 entries, 0 to 26279
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   fecha_hora          26280 non-null  object 
 1   demanda_prevista    26172 non-null  float64
 2   demanda_programada  26172 non-null  float64
 3   demanda_real        26178 non-null  float64
 4   precio_spot         26280 non-null  float64
 5   pvpc                22657 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.2+ MB


Se observa que la variable `fecha_hora` viene en formato `object` en vez de `datetime`, por lo que conviene realizar una conversión, especialmente para poder tener los datos ordenados por fecha. En principio, en el head() se muestra que siguen un formato ISO 8601 con la fecha (año-mes-día), la hora (horas:minutos:segundos) y un desfase respecto a UCT, pero habrá que comprobar si realmente todos los valores en `datetime` siguen este formato, además de comprobar si hay nulos y posteriormente cuando se realiza la conversión analizar si hay fechas duplicadas o temporalmente ausentes.

In [72]:
nulos = df_horario.isnull().sum()
print("Valores nulos por columna en df_horario:")
print(nulos)

Valores nulos por columna en df_horario:
fecha_hora               0
demanda_prevista       108
demanda_programada     108
demanda_real           102
precio_spot              0
pvpc                  3623
dtype: int64


Al encontrarse valores nulos en el dataset de horario, no se debería hacer una conversión a días directamente en el preprocesamiento sin haber tratado estos nulos previamente, ya que al realizar la suma se perdería información en las horas faltantes, lo que resulta crítico en `demanda_real`, que es la variable objetivo. Hay 108 nulos en `demanda_prevista`, 108 nulos en `demanda_programada`, 102 nulos en `demanda_real` y 3623 nulos en `pvpc`. En cuanto a la conversión de `fecha_hora`, no presenta fechas ausentes a priori. Falta comprobar si el formato ISO 8601 se cumple para todos los valores.

In [73]:
print("Distribución de la longitud (en caracteres) de los datos en 'fecha_hora'\n:" + str(df_horario['fecha_hora'].astype(str).str.len().value_counts()))
print("Desfases encontrados:\n" + str(df_horario['fecha_hora'].astype(str).str[-6:].value_counts()))

fechas_test = pd.to_datetime(df_horario['fecha_hora'], errors='coerce', utc=True)
filas_problematicas = df_horario[fechas_test.isna() & df_horario['fecha_hora'].notna()]

if len(filas_problematicas) > 0:
    print(f"Hay {len(filas_problematicas)} filas que no se pueden convertir a datetime.")
else:
    print("Todas las filas se pueden convertir a datetime correctamente.")

Distribución de la longitud (en caracteres) de los datos en 'fecha_hora'
:fecha_hora
25    26280
Name: count, dtype: int64
Desfases encontrados:
fecha_hora
+02:00    15624
+01:00    10656
Name: count, dtype: int64
Todas las filas se pueden convertir a datetime correctamente.


Todos los registros de `fecha_hora` tienen exactamente 25 caracteres y ninguno falla al convertirse a `datetime`. Además, aparecen dos desfases distintos (`+02:00` en 15.624 filas, siendo el horario de verano, y `+01:00` en 10.656 filas, siendo el horario de invierno), lo que confirma que en el dataset ya vienen reflejados los cambios entre horario de invierno y verano del sistema peninsular. Con esto ya se puede pasar la columna a `datetime` y revisar después si hay duplicados, huecos temporales o días con más o menos de 24 horas cuando se agrupen por día.

In [74]:
df_horario['fecha_hora'] = pd.to_datetime(df_horario['fecha_hora'], utc=True).dt.tz_convert('Europe/Madrid')
df_horario = df_horario.sort_values('fecha_hora')
df_horario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26280 entries, 0 to 26279
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype                        
---  ------              --------------  -----                        
 0   fecha_hora          26280 non-null  datetime64[ns, Europe/Madrid]
 1   demanda_prevista    26172 non-null  float64                      
 2   demanda_programada  26172 non-null  float64                      
 3   demanda_real        26178 non-null  float64                      
 4   precio_spot         26280 non-null  float64                      
 5   pvpc                22657 non-null  float64                      
dtypes: datetime64[ns, Europe/Madrid](1), float64(5)
memory usage: 1.2 MB


Al pasar a `datetime` se convierte a `Europe/Madrid` en lugar de dejarlo en UTC para no desajustar la serie frente a la fecha diaria de `ree_generacion_diaria.csv`, además de que permite analizar el comportamiento de la demanda y los precios según la hora real en España.

In [75]:
# Comprobar si hay fechas duplicadas
print(f"Hay {df_horario['fecha_hora'].duplicated().sum()} fechas duplicadas.")

# Comprobar los huecos temporales (en horas)
rango_ideal = pd.date_range(start=df_horario['fecha_hora'].min(), end=df_horario['fecha_hora'].max(), freq='1h')
horas_faltantes = rango_ideal.difference(df_horario['fecha_hora'])
print(f"\nHay {len(horas_faltantes)} horas faltantes en el rango temporal.")
if len(horas_faltantes) > 0:
    for hora in horas_faltantes:
        print(hora)

# Comprobar qué días están incompletos
conteo_por_dia = df_horario.groupby(df_horario['fecha_hora'].dt.date).size()
dias_incompletos = conteo_por_dia[conteo_por_dia != 24]
print(f"\nHay {len(dias_incompletos)} días incompletos:")
for dia, num_horas in dias_incompletos.items():
    print(f"{dia}: {num_horas} horas registradas")

# Primer y último registro
print(f"\nPrimer registro: {df_horario['fecha_hora'].min()}")
print(f"Último registro: {df_horario['fecha_hora'].max()}")

Hay 0 fechas duplicadas.

Hay 0 horas faltantes en el rango temporal.

Hay 6 días incompletos:
2021-03-28: 23 horas registradas
2021-10-31: 25 horas registradas
2022-03-27: 23 horas registradas
2022-10-30: 25 horas registradas
2023-03-26: 23 horas registradas
2023-10-29: 25 horas registradas

Primer registro: 2021-01-01 00:00:00+01:00
Último registro: 2023-12-31 23:00:00+01:00


No hay fechas duplicadas ni horas faltantes en todo el rango temporal, que va desde 2021-01-01 00:00:00+01:00 hasta 2023-12-31 23:00:00+01:00, y cubre todos los días en el rango temporal, sin haber ningún día ausente. En cuanto a los días incompletos, no se encuentra ninguno debido a pérdida de datos, puesto que los seis días con 23 o 25 horas registradas corresponden a los cambios de hora de verano e invierno en hora peninsular y no a huecos en los datos, algo que convendrá tener en cuenta al pasar a diario. Tampoco el primer registro o el último registro aparecen como incompletos, por lo que el dataset contiene la información completa de ambos días.

In [78]:
pct_nulos = (df_horario.isnull().sum() / len(df_horario)) * 100
df_nulos = pd.DataFrame(
    {"Nulos": nulos, "Porcentaje (%)": pct_nulos.round(2)}
).query("Nulos > 0")
if len(df_nulos) > 0:
    print(df_nulos.sort_values(by="Porcentaje (%)", ascending=False))
else:
    print("No se han detectado valores nulos explícitos (NaN).")

                    Nulos  Porcentaje (%)
pvpc                 3623           13.79
demanda_prevista      108            0.41
demanda_programada    108            0.41
demanda_real          102            0.39


Ahora que se ha confirmado que no hay huecos temporales en el rango temporal del dataset, se puede comprobar el porcentaje de valores nulos por columna de cara a decidir en un futuro preprocesamiento si se deberían imputar, conservar o descartar. Los resultados muestran en general un número de valores ausentes muy reducido. Sin embargo, `pvpc` contiene un porcentaje considerable de valores nulos, siendo del 13.79%, por lo que habrá que estudiar esta variable con mayor detenimiento para tomar una decisión en el preprocesamiento sobre cómo tratar dicha columna.